# Multispectral EuroSAT CNN on Google Colab

This notebook is designed to run in **Google Colab**, not local Jupyter/PyCharm. Select a GPU runtime before running it. It trains the 13-band Sentinel-2 ResNet18 and writes `reports/spectral_cnn_report.txt`.

In [ ]:
# This notebook must be opened at https://colab.research.google.com/.
try:
    import google.colab  # noqa: F401
except ImportError as error:
    raise RuntimeError(
        'Open this notebook in Google Colab, not local Jupyter or PyCharm.'
    ) from error

# In Colab: Runtime -> Change runtime type -> T4 GPU, then restart the session.
import subprocess
import torch

print('PyTorch:', torch.__version__)
print('CUDA compiled into PyTorch:', torch.backends.cuda.is_built())
print('CUDA available:', torch.cuda.is_available())
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except (FileNotFoundError, subprocess.CalledProcessError):
    print('nvidia-smi is unavailable: this runtime has no visible NVIDIA GPU.')
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU unavailable. Select Runtime > Change runtime type > T4 GPU, '
        'save, restart the session, and run this cell again.'
    )
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/DariusSasarman/Land-cover-classification-ROSPIN-Summer-School.git'
REPO_DIR = '/content/Land-cover-classification-ROSPIN-Summer-School'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install the packages needed by the multispectral loader and trainer.
!pip install -q rasterio scikit-learn requests

In [ ]:
# Downloads and extracts the 13-band EuroSAT GeoTIFF dataset.
# The archive is large; this can take several minutes.
!python -u src/download_eurosat_allbands.py

In [ ]:
# The trainer must be present in the cloned repository.
# If it has not been pushed to GitHub yet, upload it from your computer.
from pathlib import Path

trainer_path = Path('src/train_spectral_cnn.py')
if not trainer_path.exists():
    from google.colab import files
    print('src/train_spectral_cnn.py is missing. Select that file in the upload dialog.')
    uploaded = files.upload()
    if 'train_spectral_cnn.py' not in uploaded:
        raise FileNotFoundError('Please upload the file named train_spectral_cnn.py.')
    trainer_path.parent.mkdir(parents=True, exist_ok=True)
    trainer_path.write_bytes(uploaded['train_spectral_cnn.py'])
print('Trainer found:', trainer_path.resolve())

In [ ]:
# Train ResNet18 and capture its complete output in the notebook.
# The captured output is also saved as a text report by the next lines.
from pathlib import Path
import subprocess

report_dir = Path('reports')
report_dir.mkdir(exist_ok=True)
command = [
    'python', '-u', 'src/train_spectral_cnn.py',
    '--epochs', '30', '--batch-size', '64', '--workers', '2',
]
result = subprocess.run(
    command,
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)
Path('reports/spectral_cnn_notebook_output.txt').write_text(
    result.stdout,
    encoding='utf-8',
)
if result.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {result.returncode}.')
print('Notebook output saved to reports/spectral_cnn_notebook_output.txt')

In [ ]:
from pathlib import Path

report_path = Path('reports/spectral_cnn_report.txt')
notebook_report_path = Path('reports/spectral_cnn_notebook_output.txt')
if not report_path.exists() and notebook_report_path.exists():
    report_path.write_text(notebook_report_path.read_text(encoding='utf-8'), encoding='utf-8')
if not report_path.exists():
    raise FileNotFoundError('Training did not finish, so no report was created.')
print(report_path.read_text(encoding='utf-8'))

In [ ]:
# Compare every report that exists in this checkout.
!python -u src/compare_models.py

In [ ]:
# Optional: download the text report to your computer.
from google.colab import files
files.download('reports/spectral_cnn_report.txt')